In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect('../data/airbnb.db', read_only=True)
print("Connected to DuckDB")

tables = con.execute("SHOW TABLES").fetchall()
print(f"Tables in database: {[t[0] for t in tables]}")

Connected to DuckDB
Tables in database: ['listings', 'reviews']


Basic row counts

In [ ]:
listings_count = con.execute("SELECT COUNT(*) FROM listings").fetchone()[0]
reviews_count = con.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]

print(f"Listings: {listings_count:,} rows")
print(f"Reviews: {reviews_count:,} rows")

# Reviews per listing (average)
avg_reviews = con.execute("""
    SELECT AVG(review_count) FROM (
        SELECT listing_id, COUNT(*) as review_count 
        FROM reviews 
        GROUP BY listing_id
    )
""").fetchone()[0]
print(f"Average reviews per listing: {avg_reviews:.1f}")


Listings: 96,182 rows
Reviews: 1,887,519 rows
Average reviews per listing: 26.3


Null counts

In [ ]:
# For key columns in listings
null_counts = con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) as id_nulls,
        SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END) as name_nulls,
        SUM(CASE WHEN host_id IS NULL THEN 1 ELSE 0 END) as host_id_nulls,
        SUM(CASE WHEN host_name IS NULL THEN 1 ELSE 0 END) as host_name_nulls,
        SUM(CASE WHEN neighbourhood IS NULL THEN 1 ELSE 0 END) as neighbourhood_nulls,
        SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END) as latitude_nulls,
        SUM(CASE WHEN longitude IS NULL THEN 1 ELSE 0 END) as longitude_nulls,
        SUM(CASE WHEN room_type IS NULL THEN 1 ELSE 0 END) as room_type_nulls,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) as price_nulls,
        SUM(CASE WHEN host_is_superhost IS NULL THEN 1 ELSE 0 END) as superhost_nulls,
        SUM(CASE WHEN review_scores_rating IS NULL THEN 1 ELSE 0 END) as rating_nulls,
        SUM(CASE WHEN last_review IS NULL THEN 1 ELSE 0 END) as last_review_nulls
    FROM listings
""").df()

null_counts

,total_rows,id_nulls,name_nulls,host_id_nulls,host_name_nulls,neighbourhood_nulls,latitude_nulls,longitude_nulls,room_type_nulls,price_nulls,superhost_nulls,rating_nulls,last_review_nulls
0,96182,0.0,0.0,0.0,5.0,50520.0,0.0,0.0,0.0,32977.0,1488.0,24533.0,24533.0


Null percentages

In [ ]:
total = null_counts['total_rows'].iloc[0]
print("NULL RATES IN LISTINGS:\n")

columns_to_check = ['id_nulls', 'name_nulls', 'host_id_nulls', 'host_name_nulls',
                    'neighbourhood_nulls', 'latitude_nulls', 'longitude_nulls',
                    'room_type_nulls', 'price_nulls', 'superhost_nulls', 
                    'rating_nulls', 'last_review_nulls']

for col in columns_to_check:
    null_count = null_counts[col].iloc[0]
    pct = (null_count / total) * 100
    col_name = col.replace('_nulls', '')
    print(f"  {col_name:25s}: {null_count:>7,} nulls ({pct:5.2f}%)")

NULL RATES IN LISTINGS:

  id                       :     0.0 nulls ( 0.00%)
  name                     :     0.0 nulls ( 0.00%)
  host_id                  :     0.0 nulls ( 0.00%)
  host_name                :     5.0 nulls ( 0.01%)
  neighbourhood            : 50,520.0 nulls (52.53%)
  latitude                 :     0.0 nulls ( 0.00%)
  longitude                :     0.0 nulls ( 0.00%)
  room_type                :     0.0 nulls ( 0.00%)
  price                    : 32,977.0 nulls (34.29%)
  superhost                : 1,488.0 nulls ( 1.55%)
  rating                   : 24,533.0 nulls (25.51%)
  last_review              : 24,533.0 nulls (25.51%)


Cardinality and data type distributions

In [ ]:
cardinality_df = con.execute("""
    SELECT 
        "column_name",
        "column_type",
        "null_percentage",
        "approx_unique"
    FROM (SUMMARIZE listings)
    ORDER BY "approx_unique" DESC
""").df()

print(f"COLUMN PROFILE SUMMARY ({len(cardinality_df)} columns):\n")
print(cardinality_df.to_string(index=False))

COLUMN PROFILE SUMMARY (75 columns):

                                 column_name column_type  null_percentage  approx_unique
                                 listing_url     VARCHAR             0.00          98230
                                          id      BIGINT             0.00          96863
                                        name     VARCHAR             0.00          91251
                                   amenities     VARCHAR             0.00          89667
                                 picture_url     VARCHAR             0.01          83271
                                 description     VARCHAR             3.55          79747
                            host_picture_url     VARCHAR             0.01          71597
                                    host_url     VARCHAR             0.00          66025
                                   longitude      DOUBLE             0.00          64309
                                     host_id      BIGINT             0.0

Price Outliers

In [ ]:
# Price distribution analysis with type CAST
price_stats = con.execute("""
    SELECT 
        MIN(CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DECIMAL)) as min_price,
        AVG(CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DECIMAL)) as avg_price,
        MEDIAN(CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DECIMAL)) as median_price,
        MAX(CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DECIMAL)) as max_price,
        COUNT(CASE WHEN price = '$0.00' THEN 1 END) as zero_prices,
        COUNT(CASE WHEN CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DECIMAL) < 10 THEN 1 END) as very_low_prices,
        COUNT(CASE WHEN CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DECIMAL) > 1000 THEN 1 END) as very_high_prices
    FROM listings
""").df()

print("PRICE DISTRIBUTION:")
print(price_stats.to_string(index=False))


PRICE DISTRIBUTION:
 min_price  avg_price  median_price  max_price  zero_prices  very_low_prices  very_high_prices
       1.0 197.150225         130.0    80000.0            0                7               604


Top 10 most expensive listings

In [ ]:
print("TOP 10 MOST EXPENSIVE LISTINGS:\n")
con.execute("""
    SELECT id, name, neighbourhood, room_type, 
           CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DECIMAL) as price_numeric
    FROM listings
    WHERE CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DECIMAL) > 0
    ORDER BY price_numeric DESC
    LIMIT 10
""").df()

TOP 10 MOST EXPENSIVE LISTINGS:



,id,name,neighbourhood,room_type,price_numeric
0,1040471243366946776,Room In Zone 1 (TOB),None,Private room,80000.0
1,1069626776402758037,Close To London Bridge,None,Private room,80000.0
2,41557986,Close To London Eye (HED),None,Private room,75000.0
3,42435181,CLOSE TO LONDON EYE (CHECZ),None,Private room,75000.0
4,1183168933886992107,Short Walk To London Eye (SUR),None,Private room,71000.0
5,42596787,Stunning Renovated Townhouse in Pretty Square,Neighborhood highlights,Entire home/apt,60000.0
6,13254774,No Longer Available,Neighborhood highlights,Private room,53588.0
7,1204543804094089153,Short Walk to London Eye (HON),None,Private room,50000.0
8,9721759,3 Bed Flat in South Hampstead with Large Garden!,Neighborhood highlights,Entire home/apt,25000.0
9,52112740,—,None,Entire home/apt,20000.0


Duplicate identification

In [22]:
# Check for duplicate listings by ID
duplicate_ids = con.execute("""
    SELECT id, COUNT(*) as duplicate_count
    FROM listings
    GROUP BY id
    HAVING COUNT(*) > 1
    ORDER BY duplicate_count DESC
    LIMIT 10
""").df()

print(f"DUPLICATE LISTING IDS:")
print(f"Total duplicate IDs: {len(duplicate_ids)}")
if len(duplicate_ids) > 0:
    print(duplicate_ids.to_string(index=False))
else:
    print("No duplicate listing IDs found")

DUPLICATE LISTING IDS:
Total duplicate IDs: 0
No duplicate listing IDs found


Geographic validation

In [23]:
# Check coordinate ranges (London should be ~51.5°N, -0.1°W)
coord_check = con.execute("""
    SELECT 
        MIN(latitude) as min_lat, MAX(latitude) as max_lat,
        MIN(longitude) as min_lon, MAX(longitude) as max_lon,
        COUNT(CASE WHEN latitude < 49 OR latitude > 61 THEN 1 END) as invalid_lat,
        COUNT(CASE WHEN longitude < -8 OR longitude > 2 THEN 1 END) as invalid_lon,
        COUNT(CASE WHEN latitude = 0 OR longitude = 0 THEN 1 END) as zero_coords
    FROM listings
""").df()

print("COORDINATE VALIDATION:")
print(coord_check.to_string(index=False))
print("\n Expected London range: ~51.4-51.7°N, -0.3 to 0.1°W")

COORDINATE VALIDATION:
  min_lat   max_lat  min_lon  max_lon  invalid_lat  invalid_lon  zero_coords
51.295937 51.681642  -0.4978 0.295731            0            0            0

 Expected London range: ~51.4-51.7°N, -0.3 to 0.1°W


In [36]:
con.close()
con = duckdb.connect('../data/airbnb.db')
print("Connected in write mode")

con.execute("""
    CREATE OR REPLACE VIEW listings_clean AS
    SELECT *,
           CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DECIMAL) as price_clean
    FROM listings
""")
print("View created")


Connected in write mode
View created


Room type distribution

In [ ]:
# Room type breakdown using clean view
room_types = con.execute("""
    SELECT 
        room_type,
        COUNT(*) as count,
        ROUND(AVG(price_clean), 2) as avg_price,
        ROUND(MIN(price_clean), 2) as min_price,
        ROUND(MAX(price_clean), 2) as max_price
    FROM listings_clean
    GROUP BY room_type
    ORDER BY count DESC
""").df()

print("ROOM TYPE DISTRIBUTION:\n")
print(room_types.to_string(index=False))

ROOM TYPE DISTRIBUTION:

      room_type  count  avg_price  min_price  max_price
Entire home/apt  61321     238.77       12.0    60000.0
   Private room  34236     111.38        1.0    80000.0
    Shared room    437     147.31        8.0     3500.0
     Hotel room    188     371.49       21.0     7589.0


Fuzzy duplicates detection

In [ ]:
# Check for similar listing names in same neighbourhood
fuzzy_dupes = con.execute("""
    SELECT 
        neighbourhood,
        name,
        price,
        host_id,
        COUNT(*) as similar_count
    FROM listings
    WHERE name IS NOT NULL AND neighbourhood IS NOT NULL
    GROUP BY neighbourhood, name, price, host_id
    HAVING COUNT(*) > 1
    ORDER BY similar_count DESC
    LIMIT 10
""").df()

print(f"FUZZY DUPLICATES (same name + neighbourhood + price + host):")
print(f"Total suspected fuzzy duplicates: {len(fuzzy_dupes)}")
if len(fuzzy_dupes) > 0:
    print(fuzzy_dupes.to_string(index=False))
else:
    print("No fuzzy duplicates found")


FUZZY DUPLICATES (same name + neighbourhood + price + host):
Total suspected fuzzy duplicates: 10
          neighbourhood                                               name   price   host_id  similar_count
Neighborhood highlights               Private Double Room in Warren Street  $57.00 268416645             10
Neighborhood highlights    Hyde Park Luxurious Hotel, Faboulas Double Room    None 345760385              9
Neighborhood highlights   Hyde Park Luxurious Hotel, Beautiful Family Room    None 345760385              7
Neighborhood highlights Moorgate Old Street 2 bedroom 2 bathroom apartment    None  47609036              6
Neighborhood highlights    Double bed in newly refurbished warm guesthouse    None 236963899              5
Neighborhood highlights             Stunning Shoreditch High Street Studio    None   5628346              4
Neighborhood highlights       Clerkenwell 1 bedroom apartment with balcony    None  47609036              4
Neighborhood highlights              C

Availability outliers

In [31]:
avail_stats = con.execute("""
    SELECT 
        MIN(availability_365) as min_avail,
        AVG(availability_365) as avg_avail,
        MEDIAN(availability_365) as median_avail,
        MAX(availability_365) as max_avail,
        COUNT(CASE WHEN availability_365 = 0 THEN 1 END) as zero_avail,
        COUNT(CASE WHEN availability_365 = 365 THEN 1 END) as full_year_avail
    FROM listings
""").df()

print("AVAILABILITY_365 DISTRIBUTION:\n")
print(avail_stats.to_string(index=False))
print(f"\n Zero availability = likely blocked/booked")
print(f"365 availability = always available (commercial listings)")

AVAILABILITY_365 DISTRIBUTION:

 min_avail  avg_avail  median_avail  max_avail  zero_avail  full_year_avail
         0 132.112817          88.0        365       30512             3097

 Zero availability = likely blocked/booked
365 availability = always available (commercial listings)


Reviews count outliers + price validation

In [34]:
review_price_check = con.execute("""
    SELECT 
        'Listings with negative price' as check_name,
        COUNT(*) as count
    FROM listings_clean WHERE price_clean < 0
    
    UNION ALL
    
    SELECT 'Listings with price = 0',
        COUNT(*)
    FROM listings_clean WHERE price_clean = 0
    
    UNION ALL
    
    SELECT 'Listings with 0 reviews',
        COUNT(*)
    FROM listings_clean WHERE number_of_reviews = 0
    
    UNION ALL
    
    SELECT 'Listings with >500 reviews',
        COUNT(*)
    FROM listings_clean WHERE number_of_reviews > 500
    
    UNION ALL
    
    SELECT 'Listings with 365 availability',
        COUNT(*)
    FROM listings_clean WHERE availability_365 = 365
""").df()

print("DOMAIN VALIDATION CHECKS:\n")
print(review_price_check.to_string(index=False))

DOMAIN VALIDATION CHECKS:

                    check_name  count
  Listings with negative price      0
       Listings with price = 0      0
       Listings with 0 reviews  24533
    Listings with >500 reviews     89
Listings with 365 availability   3097


In [40]:
con.close()
print("Connection closed!")

Connection closed!
